# Notebook 01b — Hand X-Ray: Balanced Oversampling & Safe Augmentation

**Goal:** Membuat dataset seimbang dari gambar X-ray tangan dengan mempertahankan kelas mayoritas dan mengaugmentasi kelas minoritas.

## Strategi
1. **Oversampling:** Pertahankan semua kelas mayoritas (Non-fractured) dan augmentasi kelas minoritas (Fractured) agar seimbang.
2. **Augmentasi Aman Secara Anatomis:** Buat salinan augmentasi untuk gambar minoritas agar total mencapai target.
3. **Stratified Split:** Bagi data menjadi Train (70%) / Val (15%) / Test (15%) dengan menjaga rasio kelas.

## Augmentation Strategy (Anatomically Safe)
✅ **SAFE - Diterapkan:**
- **Horizontal Flip** (p=0.5) — Aman untuk X-ray tangan (simetri kiri ↔ kanan)
- **Small Rotation** (±10°) — Mensimulasikan variasi posisi pasien
- **Brightness/Contrast** — Mensimulasikan kualitas mesin X-ray berbeda
- **Mild Translation** (5%) — Perbedaan posisi tangan
- **Mild Scale** (±15%) — Perbedaan ukuran tangan / jarak

❌ **UNSAFE - Dinonaktifkan:**
- **Vertical Flip** — Membuat anatomi tidak benar (tulang terbalik)
- **Shear** — Mendistorsi struktur tulang
- **Perspective** — X-ray adalah proyeksi, bukan perspektif
- **Hue/Saturation** — X-ray bersifat grayscale

## 0. Setup & Konfigurasi

In [1]:
import os
import cv2
import random
import shutil
import math
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import albumentations as A
import yaml

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR   = Path.cwd().parent
IMAGES_DIR = BASE_DIR / "images"
ANNOT_DIR  = BASE_DIR / "Annotations" / "YOLO"
DATASET_CSV = BASE_DIR / "dataset.csv"
OUTPUT_DIR  = BASE_DIR / "yolo_dataset_hand_oversampled"

# ── Split Ratio ──────────────────────────────────────────────────────────────
VAL_RATIO      = 0.15
TEST_RATIO     = 0.15
TRAIN_RATIO    = 0.70

# ── Create Output Structure ──────────────────────────────────────────────────
for split in ["train", "val", "test"]:
    (OUTPUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

print(f"Base directory  : {BASE_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Split ratio     : Train {TRAIN_RATIO:.0%} | Val {VAL_RATIO:.0%} | Test {TEST_RATIO:.0%}")

Base directory  : d:\Project Medical Object Detection\FracAtlas
Output directory: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_hand_oversampled
Split ratio     : Train 70% | Val 15% | Test 15%


## 1. Load & Analisis Dataset

In [2]:
# Load dan filter hanya gambar tangan
df = pd.read_csv(DATASET_CSV)
hand_df = df[df['hand'] == 1].copy()

n_frac     = len(hand_df[hand_df['fractured'] == 1])
n_non_frac = len(hand_df[hand_df['fractured'] == 0])
n_total    = len(hand_df)

print("=" * 50)
print("DISTRIBUSI DATA HAND (SEBELUM BALANCING)")
print("=" * 50)
print(f"  Fractured        : {n_frac:>6}  ({n_frac/n_total*100:.1f}%)")
print(f"  Non-fractured    : {n_non_frac:>6}  ({n_non_frac/n_total*100:.1f}%)")
print(f"  Total Hand Images: {n_total:>6}")
print("=" * 50)

# Tampilkan kolom untuk validasi
print(f"\nKolom tersedia: {list(df.columns)}")
print(f"Sample data:")
print(hand_df.head(3))

DISTRIBUSI DATA HAND (SEBELUM BALANCING)
  Fractured        :    438  (28.5%)
  Non-fractured    :   1100  (71.5%)
  Total Hand Images:   1538

Kolom tersedia: ['image_id', 'hand', 'leg', 'hip', 'shoulder', 'mixed', 'hardware', 'multiscan', 'fractured', 'fracture_count', 'frontal', 'lateral', 'oblique']
Sample data:
          image_id  hand  leg  hip  shoulder  mixed  hardware  multiscan  \
6   IMG0000006.jpg     1    0    0         0      0         0          1   
7   IMG0000007.jpg     1    0    0         0      0         0          0   
11  IMG0000011.jpg     1    0    0         1      1         0          0   

    fractured  fracture_count  frontal  lateral  oblique  
6           0               0        0        1        1  
7           0               0        0        0        1  
11          0               0        1        0        0  


## 2. Dynamic Oversampling

In [3]:
# Pada oversampling, kita tidak membuang data mayoritas
# Kita gunakan SEMUA data hand_df
balanced_df = hand_df.copy()

# Hitung berapa kali kelas minoritas (fracture) perlu di-augmentasi agar seimbang
n_copies_needed = math.ceil(n_non_frac / n_frac)

print("=" * 50)
print("HASIL CONFIG OVERSAMPLING")
print("=" * 50)
print(f"  Majority count (Non-frac) : {n_non_frac}")
print(f"  Minority count (Frac)     : {n_frac}")
print(f"  Multiplier for Fracture   : {n_copies_needed}x (1 original + {n_copies_needed-1} augmented)")
print()
print("=" * 50)
print("RENCANA AUGMENTASI")
print("=" * 50)
print(f"  Total base images      : {n_total}")
print(f"  Estimated total output : {n_non_frac + (n_frac * n_copies_needed)}")
print(f"    → Non-frac total     : {n_non_frac} (Original only)")
print(f"    → Fractured total    : ~{n_frac * n_copies_needed} (Original + Augmented)")

HASIL CONFIG OVERSAMPLING
  Majority count (Non-frac) : 1100
  Minority count (Frac)     : 438
  Multiplier for Fracture   : 3x (1 original + 2 augmented)

RENCANA AUGMENTASI
  Total base images      : 1538
  Estimated total output : 2414
    → Non-frac total     : 1100 (Original only)
    → Fractured total    : ~1314 (Original + Augmented)


## 3. Define Anatomically Safe Augmentation Pipeline

In [4]:
# Pipeline augmentasi yang sama persis dengan versi undersampling
aug_pipeline = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.Affine(
            rotate=(-10, 10),
            translate_percent=(-0.05, 0.05),
            scale=(0.85, 1.15),
            shear=0,
            mode=cv2.BORDER_CONSTANT,
            cval=0,
            p=0.8
        ),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.005, 0.02), p=0.2)
    ],
    bbox_params=A.BboxParams(
        format='yolo',
        label_fields=['class_labels'],
        min_visibility=0.3
    )
)

print("Augmentation pipeline siap (Same Configuration).")

Augmentation pipeline siap (Same Configuration).


C:\Users\alema\AppData\Local\Temp\ipykernel_29320\1718287722.py:5: UserWarning: Argument(s) 'mode, cval' are not valid for transform Affine
  A.Affine(


## 4. Helper Functions

In [5]:
def get_image_and_label_paths(img_id: str):
    p_frac     = IMAGES_DIR / "Fractured"     / img_id
    p_non_frac = IMAGES_DIR / "Non_fractured" / img_id
    img_path   = p_frac if p_frac.exists() else p_non_frac
    
    base_name  = Path(img_id).stem
    label_path = ANNOT_DIR / f"{base_name}.txt"
    return img_path, label_path

def read_yolo_labels(label_path: Path):
    bboxes, class_labels = [], []
    if label_path.exists():
        with open(label_path, 'r') as f:
            for line in f:
                vals = list(map(float, line.split()))
                if len(vals) == 5:
                    class_labels.append(int(vals[0]))
                    bboxes.append(vals[1:])
    return bboxes, class_labels

def save_image_and_label(image_rgb, bboxes, class_labels, out_img_path, out_lbl_path):
    cv2.imwrite(str(out_img_path), cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
    with open(out_lbl_path, 'w') as f:
        for cls, bbox in zip(class_labels, bboxes):
            f.write(f"{cls} {' '.join(map(str, bbox))}\n")

## 5. Generate Augmented Dataset

In [6]:
# Bersihkan folder
for split in ["train", "val", "test"]:
    for f in (OUTPUT_DIR / split / "images").glob("*.jpg"): f.unlink()
    for f in (OUTPUT_DIR / split / "labels").glob("*.txt"): f.unlink()

skipped, total_saved = 0, 0

for _, row in tqdm(balanced_df.iterrows(), total=len(balanced_df), desc="Generating images"):
    img_id = row['image_id']
    is_fractured = row['fractured'] == 1
    img_path, label_path = get_image_and_label_paths(img_id)

    if not img_path.exists():
        skipped += 1
        continue

    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    bboxes, classes = read_yolo_labels(label_path)

    # ── Simpan Original (Ke folder TRAIN untuk sementara) ──────────────────────
    out_img = OUTPUT_DIR / "train" / "images" / f"{Path(img_id).stem}_orig.jpg"
    out_lbl = OUTPUT_DIR / "train" / "labels" / f"{Path(img_id).stem}_orig.txt"
    save_image_and_label(img_rgb, bboxes, classes, out_img, out_lbl)
    total_saved += 1

    # ── Simpan Augmented Copies (HANYA untuk Fracture) ───────────────────────
    if is_fractured:
        for i in range(n_copies_needed - 1):
            try:
                aug = aug_pipeline(image=img_rgb, bboxes=bboxes, class_labels=classes)
                out_img_a = OUTPUT_DIR / "train" / "images" / f"{Path(img_id).stem}_aug{i}.jpg"
                out_lbl_a = OUTPUT_DIR / "train" / "labels" / f"{Path(img_id).stem}_aug{i}.txt"
                save_image_and_label(aug['image'], aug['bboxes'], aug['class_labels'], out_img_a, out_lbl_a)
                total_saved += 1
            except: pass

print("\n" + "=" * 50)
print("HASIL GENERASI DATA (OVERSAMPLING)")
print("=" * 50)
print(f"  Gambar tersimpan  : {total_saved}")
print(f"  Gambar di-skip    : {skipped}")

Generating images: 100%|██████████| 1538/1538 [00:32<00:00, 47.67it/s] 


HASIL GENERASI DATA (OVERSAMPLING)
  Gambar tersimpan  : 2414
  Gambar di-skip    : 0


## 6. Stratified Train / Val / Test Split

In [7]:
# Karena oversampling dilakukan per gambar, kita pastikan semua versi (orig+aug) dari 1 image_id
# masuk ke split yang sama agar tidak ada data leakage.
all_imgs = sorted((OUTPUT_DIR / "train" / "images").glob("*.jpg"))

frac_samples = hand_df[hand_df['fractured'] == 1]
non_frac_samples = hand_df[hand_df['fractured'] == 0]

frac_imgs = [p for p in all_imgs if any(p.stem.startswith(Path(iid).stem) for iid in frac_samples['image_id'].values)]
non_frac_imgs = [p for p in all_imgs if any(p.stem.startswith(Path(iid).stem) for iid in non_frac_samples['image_id'].values)]

random.shuffle(frac_imgs)
random.shuffle(non_frac_imgs)

def split_list(lst, val_r, test_r):
    n_val, n_test = int(len(lst) * val_r), int(len(lst) * test_r)
    return lst[n_val + n_test:], lst[:n_val], lst[n_val:n_val + n_test]

f_train, f_val, f_test = split_list(frac_imgs, VAL_RATIO, TEST_RATIO)
n_train, n_val, n_test = split_list(non_frac_imgs, VAL_RATIO, TEST_RATIO)

def move_split(img_list, target_split):
    for img_p in img_list:
        if target_split == "train": continue
        lbl_src = OUTPUT_DIR / "train" / "labels" / f"{img_p.stem}.txt"
        img_dst = OUTPUT_DIR / target_split / "images" / img_p.name
        lbl_dst = OUTPUT_DIR / target_split / "labels" / f"{img_p.stem}.txt"
        shutil.move(str(img_p), str(img_dst))
        if lbl_src.exists(): shutil.move(str(lbl_src), str(lbl_dst))

move_split(f_val + n_val, "val")
move_split(f_test + n_test, "test")

final_train = len(list((OUTPUT_DIR / "train" / "images").glob("*.jpg")))
final_val   = len(list((OUTPUT_DIR / "val" / "images").glob("*.jpg")))
final_test  = len(list((OUTPUT_DIR / "test" / "images").glob("*.jpg")))
total_f     = final_train + final_val + final_test

print("=" * 50)
print("DISTRIBUSI FINAL DATASET (OVERSAMPLED)")
print("=" * 50)
print(f"  Train : {final_train:>5} gambar  ({final_train/total_f*100:.1f}%)")
print(f"  Val   : {final_val:>5} gambar  ({final_val/total_f*100:.1f}%)")
print(f"  Test  : {final_test:>5} gambar  ({final_test/total_f*100:.1f}%)")
print(f"  Total : {total_f:>5} gambar")

DISTRIBUSI FINAL DATASET (OVERSAMPLED)
  Train :  1690 gambar  (70.0%)
  Val   :   362 gambar  (15.0%)
  Test  :   362 gambar  (15.0%)
  Total :  2414 gambar


## 7. Generate YAML Config untuk YOLOv8

In [8]:
yaml_config = {
    'path': str(OUTPUT_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    1,
    'names': ['fracture']
}

yaml_path = OUTPUT_DIR / "dataset.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False)

print(f"YAML config disimpan: {yaml_path}")

YAML config disimpan: d:\Project Medical Object Detection\FracAtlas\yolo_dataset_hand_oversampled\dataset.yaml


## 8. Verifikasi Akhir

In [9]:
print("=" * 60)
print("VERIFIKASI KONSISTENSI DATASET OVERSAMPLED")
print("=" * 60)
for split in ["train", "val", "test"]:
    imgs   = len(list((OUTPUT_DIR / split / "images").glob("*.jpg")))
    labels = len(list((OUTPUT_DIR / split / "labels").glob("*.txt")))
    print(f"[{split.upper()}] Images: {imgs}, Labels: {labels} {'✅' if imgs==labels else '❌'}")

print("\nDataset siap digunakan untuk training perbandingan!")

VERIFIKASI KONSISTENSI DATASET OVERSAMPLED
[TRAIN] Images: 1690, Labels: 1690 ✅
[VAL] Images: 362, Labels: 362 ✅
[TEST] Images: 362, Labels: 362 ✅

Dataset siap digunakan untuk training perbandingan!
